# Tutorial 3 — WorldView-3, Mount Soledad, three images blended

> Reading this on the docs site? To run it, [open a Codespace](https://stereopipeline-quickstart.readthedocs.io/en/latest/start/codespaces.html): ASP and `asp-plot` are already installed there.

[Tutorial 2](02_worldview_ucsd.ipynb) made a DEM from one pair of images. With a third image of the same place there are three pairs, and three DEMs that each miss different spots and carry different noise. Averaging them gives one DEM with fewer holes and less noise; on this image stack, that beat any single pair. This tutorial runs stereo on all three pairs of the same 3 × 3 km window and blends the results, in about 16 minutes after Tutorial 2 (about 20 on its own).

> To run it, click **Run All** at the top, then scroll down to follow it: each step shows its progress while it runs, then its results.

| Image | Date | Looking |
|---|---|---|
| `104001000496A100` | 2014-11-09 | 1.1° from straight down |
| `10400100071D8800` | 2015-01-24 | 16.3° to the side |
| `10400100057DD500` | 2014-12-23 | 24.1° to the side |

The steps:

1. Download the three images (two are reused from Tutorial 2).
2. Look at the viewing geometry.
3. Crop all three to the study area.
4. Refine the three cameras together (bundle adjustment).
5. Run stereo matching on each pair, for three DEMs.
6. Blend the three DEMs into one.
7. Make a PDF report of the blended DEM, with a comparison against ICESat-2.
8. Download the results.

In [ ]:
from pathlib import Path

from utils import run, run_stereo, timer

DATA = "../data/ucsd_soledad"
LOGS = f"{DATA}/logs"

!mkdir -p {DATA}

# The processing window, xmin ymin xmax ymax in UTM zone 11N (EPSG:32611)
# meters: 3 x 3 km around the summit of Mount Soledad.
T_SRS = "EPSG:32611"
T_PROJWIN = "476000 3632000 479000 3635000"

# The Copernicus reference DEM, fetched by the download step.
REF_DEM = f"{DATA}/ref/cop30_soledad.tif"

# Crops are averaged FACTOR x FACTOR pixels: 4 turns the native ~0.31 m
# pixels into ~1.2 m ones, which keeps stereo to a few minutes.
FACTOR = 4

SCENES = ["104001000496A100", "10400100071D8800", "10400100057DD500"]
PAIRS = [(SCENES[0], SCENES[1]), (SCENES[0], SCENES[2]), (SCENES[2], SCENES[1])]

FRESH_ICESAT2 = False
SHIPPED_SELECTIONS = f"{DATA}/icesat2_shipped/blend_figure_selections.yml"

## 1. Download the images

The same images as Tutorial 2 plus a third; images already downloaded are skipped. They are IARPA CORE3D data, released for public research; publications that use them should cite Hagstrom et al. (2021), "Cumulative assessment for urban 3D modeling" (IEEE IGARSS).

> Under a minute after Tutorial 2, or about 2 minutes without it.

In [ ]:
run(f"bash ../scripts/download_worldview_ucsd.sh {DATA} {' '.join(SCENES)}", log=f"{LOGS}/download.log", label="Downloading the images")

## 2. Viewing geometry

[More on viewing geometry](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/stereo-photogrammetry.html#viewing-geometry).

> About half a minute.

In [ ]:
with timer("Plotting the stereo geometry"):
    import contextily
    from asp_plot.stereo_geometry import StereoGeometryPlotter

    # The map tiles were fetched when the Codespace was built
    # (scripts/prefetch_basemap.py); without them, this fetches them itself.
    contextily.set_cache_dir("../data/basemap_cache")

    geometry = StereoGeometryPlotter(inputs=[f"{DATA}/{catid}_P001.xml" for catid in SCENES])
    geometry.stereo_geom_plot()

The three view directions spread around the sky: one near straight down and two from different sides. Each pair sees the ground from a different angle, so a spot hidden from one pair is often visible to another.

## 3. Crop to the study area

The same cut and averaging as Tutorial 2, for all three images; crops already made are skipped.

> Under a minute after Tutorial 2, or about 2 minutes without it.

In [ ]:
from utils import average_crop

Path(f"{DATA}/crops/full").mkdir(parents=True, exist_ok=True)
xmin, ymin, xmax, ymax = map(float, T_PROJWIN.split())
RPC_BOX = f"{DATA}/crops/rpc_box.tif"
if not Path(RPC_BOX).exists():
    # The reference DEM over the window plus 250 m: the ground cam2rpc fits over.
    !gdal_translate -q -projwin {xmin - 250} {ymax + 250} {xmax + 250} {ymin - 250} {REF_DEM} {RPC_BOX}

for catid in SCENES:
    if Path(f"{DATA}/crops/{catid}_P001.tif").exists():
        print(f"{DATA}/crops/{catid}_P001.tif exists — skipping. Delete it to recrop.")
        continue
    run(f"""
        cam2rpc -t dg --dem-file {RPC_BOX} --save-tif-image \
            {DATA}/{catid}_P001.NTF {DATA}/{catid}_P001.xml \
            {DATA}/crops/full/{catid}.xml
        """,
        log=f"{LOGS}/cam2rpc_{catid}.log",
        label=f"Cropping {catid}",
    )
    with timer(f"Averaging {catid} to {FACTOR} x {FACTOR} pixels"):
        average_crop(f"{DATA}/crops/full/{catid}.tif", f"{DATA}/crops/full/{catid}.xml",
                     f"{DATA}/crops/{catid}_P001.tif", f"{DATA}/crops/{catid}_P001.xml", FACTOR)

!ls -lh {DATA}/crops/*.tif

## 4. Bundle adjustment

All three cameras are refined together, so that every pair is measured with cameras that agree. [What bundle adjustment does](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/bundle-adjustment.html).

> Under a minute.

In [ ]:
if Path(f"{DATA}/ba3/run-final_residuals_pointmap.csv").exists():
    print(f"{DATA}/ba3/run-final_residuals_pointmap.csv exists — skipping bundle_adjust. Delete {DATA}/ba3/ to reprocess.")
else:
    images = " ".join(f"{DATA}/crops/{c}_P001.tif" for c in SCENES)
    cameras = " ".join(f"{DATA}/crops/{c}_P001.xml" for c in SCENES)
    run(f"""
        bundle_adjust -t rpc --threads 4 \
            --heights-from-dem {Path(REF_DEM).resolve()} --heights-from-dem-uncertainty 10 \
            {images} {cameras} -o {DATA}/ba3/run
        """,
        log=f"{LOGS}/bundle_adjust_3.log",
        label="Bundle adjusting the three cameras",
    )

!cat {DATA}/ba3/run-initial_residuals_stats.txt {DATA}/ba3/run-final_residuals_stats.txt

The reprojection error per image, before and after: the median drops from about a pixel to about 0.2 pixels for each of the three.

## 5. Stereo matching, pair by pair

Stereo runs on the crops directly, without orthorectifying them first; on crops this small that works as well. For each pair, the cell shows the matched features, then the map of the area with the offsets between the images and the heights, as it fills in. [How matching works](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/stereo-photogrammetry.html).

> About 15 minutes, 3½ to 5½ minutes per pair.

In [ ]:
def pair_dir(left, right):
    return f"{DATA}/pairs/{left}_{right}"

for n, (left, right) in enumerate(PAIRS, start=1):
    out = pair_dir(left, right)
    if Path(f"{out}/run-DEM.tif").exists():
        print(f"{out}/run-DEM.tif exists — skipping. Delete {out}/ to reprocess.")
        continue
    run_stereo(f"""
        parallel_stereo -t rpc \
            --stereo-algorithm asp_mgm --subpixel-mode 9 \
            --job-size-w 4096 --job-size-h 4096 \
            --processes 1 --threads-multiprocess 4 --threads-singleprocess 4 \
            --bundle-adjust-prefix {DATA}/ba3/run \
            {DATA}/crops/{left}_P001.tif {DATA}/crops/{right}_P001.tif \
            {DATA}/crops/{left}_P001.xml {DATA}/crops/{right}_P001.xml \
            {out}/run
        """,
        prefix=f"{out}/run",
        log=f"{LOGS}/parallel_stereo_pair{n}.log",
        label=f"Stereo, pair {n} of 3",
    )
    run(
        f"point2dem --threads 4 --tr 2.4 --t_srs {T_SRS} --errorimage {out}/run-PC.tif",
        log=f"{LOGS}/point2dem_pair{n}.log",
        label=f"Gridding pair {n} into a DEM",
    )

For each pair: the matched features (red), then the offset between the two input images at every pixel, split into its parts along the image's rows and columns, and the heights triangulated from them (right). The offsets are tens of pixels here: the images weren't orthorectified, so the full [relief displacement](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/orthorectification.html) from the different viewing angles remains, plus any error left in the cameras. The map is in the first image's own orientation.

In [ ]:
from utils import show_hillshades

show_hillshades([f"{pair_dir(l, r)}/run-DEM.tif" for l, r in PAIRS],
                ["nadir + 16°", "nadir + 24°", "24° + 16°"])

The three pair DEMs as hillshades. They agree on the landforms, but each has its own holes (white) and its own speckle. The two wider pairs see the steep canyon walls from more of an angle and show more texture there.

## 6. Blend the three DEMs

ASP's `dem_mosaic` averages the three DEMs pixel by pixel. Where one DEM has a hole, the others fill it.

> A few seconds.

In [ ]:
BLEND = f"{DATA}/pairs/{PAIRS[0][0]}_{PAIRS[0][1]}/run-blend-DEM.tif"
if Path(BLEND).exists():
    print(f"{BLEND} exists — skipping dem_mosaic. Delete it to reblend.")
else:
    dems = " ".join(f"{pair_dir(l, r)}/run-DEM.tif" for l, r in PAIRS)
    run(f"dem_mosaic --threads 4 {dems} -o {BLEND}", log=f"{LOGS}/dem_mosaic.log", label="Blending the three DEMs")

show_hillshades([f"{pair_dir(*PAIRS[0])}/run-DEM.tif", BLEND],
                ["One pair (Tutorial 2's, here without orthorectification)", "Three pairs blended"])

Left, one pair; right, the blend. The blend fills nearly every hole and the speckle averages out, while ridges, roofs and roads stay sharp.

## 7. Report

`asp_report`, from `asp-plot`, writes a PDF with everything about the run, here with the blended DEM as its result: the images, the viewing geometry, the matches, the DEM, and how it compares with the reference map and with [ICESat-2](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/alignment.html), a satellite that measures heights directly with a laser. [What's in the report](https://stereopipeline-quickstart.readthedocs.io/en/latest/concepts/visualization.html). Three of its figures appear below.

> About a minute.

In [ ]:
from utils import show_report_figures

STEREO = f"pairs/{PAIRS[0][0]}_{PAIRS[0][1]}"
FIGS = f"{DATA}/{STEREO}/report_figures"
if Path(f"{DATA}/{STEREO}/soledad_wv3_blend_report.pdf").exists():
    print(f"{DATA}/{STEREO}/soledad_wv3_blend_report.pdf exists — skipping asp_report. Delete it to regenerate.")
else:
    reuse = "" if FRESH_ICESAT2 else f"--reuse-selections {SHIPPED_SELECTIONS}"
    run(f"""
        asp_report \
            --directory {DATA} \
            --bundle-adjust-prefix ba3 \
            --stereo-directory {STEREO} \
            --dem-filename run-blend-DEM.tif \
            --reference-dem {REF_DEM} \
            --subset-km 0.5 \
            {reuse} \
            --figures-directory {FIGS} \
            --report-filename soledad_wv3_blend_report.pdf
        """,
        log=f"{LOGS}/asp_report_blend.log",
        label="Making the report",
    )

show_report_figures(FIGS)

## 📄 [Open the report (PDF)](../data/ucsd_soledad/pairs/104001000496A100_10400100071D8800/soledad_wv3_blend_report.pdf)

It opens in a tab here. To keep a copy, right-click the file in the explorer on the left and choose **Download**. Compare it with Tutorial 2's report: the ICESat-2 profile follows the same track.

## 8. Download the results

The cell below puts the report and the blended DEM into one zip file, `soledad_blend_results.zip`, in the top folder of the file explorer. The GeoTIFFs open in QGIS, ArcGIS or any other GIS. When the zip is ready, a notification at the bottom right points to it; right-click it in the explorer and choose **Download**.

> A few seconds.

Each pair's DEM and map of triangulation error stay in `data/ucsd_soledad/pairs/`; they can be downloaded one by one the same way.

In [ ]:
from utils import zip_results

zip_results("../soledad_blend_results.zip", [f"{DATA}/{STEREO}/soledad_wv3_blend_report.pdf", BLEND])

### How to download it

The zip is in the **file explorer**, the panel on the left of the window with `data/`, `notebooks/` and the zip (if the panel is hidden, click the top icon in the bar at the far left). Right-click the zip there and choose **Download**; your browser then saves it like any download.

![The end of Tutorial 1: (1) the report link, (2) the results line, (3) the zip in the file explorer](https://stereopipeline-quickstart.readthedocs.io/en/latest/_images/launch-done.png)

The first time you right-click in the explorer, your browser may ask whether this page can see your clipboard. Click **Allow** or **Block**, whichever you prefer; the menu with **Download** works either way.

![The browser's clipboard question, with Block and Allow buttons](https://stereopipeline-quickstart.readthedocs.io/en/latest/_images/clipboard-prompt.png)